# 02: SEC EDGAR Fundamentals

Pulls financial data and text from SEC filings (10-K, 10-Q, 20-F) for the companies in `analysis/financials/tickers.csv`.

| Section | Question it helps answer |
|---|---|
| 1. Annual financials | Are intermediaries' revenue and margins holding up? |
| 2. Marketing intensity | Are they spending more to win customers (e.g. as search traffic shifts to AI)? |
| 3. Recent quarters | What does the latest trend look like? |
| 4. AI mentions in filings | When did companies start flagging AI agents as a risk or opportunity? |
| 5. Explore & read | Find other metrics; jump straight to the filing documents |

All data comes from the free SEC APIs via `analysis/financials/edgar.py`. Raw responses are cached in `analysis/data/raw/edgar/`, so re-running is fast. Pass `refresh=True` to re-download.

## 0. Setup
**The SEC requires you to identify yourself.** Put your name and email below (they're sent only to sec.gov).

In [ ]:
SEC_USER_AGENT = "Your Name your.email@school.edu"   # TODO: fill in

import sys, os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "analysis").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis" / "financials"))
import edgar

if "your.email" not in SEC_USER_AGENT:
    edgar.set_user_agent(SEC_USER_AGENT)
elif not os.environ.get("SEC_USER_AGENT"):
    raise RuntimeError("Fill in SEC_USER_AGENT above first.")

PROCESSED = ROOT / "analysis" / "data" / "processed"
FIG = ROOT / "analysis" / "figures"
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

tickers = pd.read_csv(ROOT / "analysis" / "financials" / "tickers.csv")
companies = tickers[tickers.sector != "benchmark"]       # ETFs don't file 10-Ks
companies[["ticker", "company", "sector", "role"]]

### Chart style
Each ticker gets a fixed color that stays the same in every chart. Recessive grid, no dual axes.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
COLOR = {}
for _, grp in companies.groupby("sector", sort=False):        # fixed order within each sector
    for i, t in enumerate(grp.ticker):
        COLOR[t] = PALETTE[i % len(PALETTE)]

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ad", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9.5, "lines.linewidth": 2,
    "legend.frameon": False,
})

def line_chart(ax, wide, title, ylabel, fmt="{:,.0f}"):
    """One line per ticker, labeled directly at its last point (labels nudged apart so they don't collide)."""
    ends = []
    for t in wide.columns:
        s = wide[t].dropna()
        if s.empty:
            continue
        ax.plot(s.index, s.values, color=COLOR.get(t, "#6b6a64"), label=t, marker="o", markersize=4)
        ends.append([s.iloc[-1], s.index[-1], f"{t} {fmt.format(s.iloc[-1])}"])
    lo, hi = ax.get_ylim()
    gap = (hi - lo) * 0.045
    ends.sort(key=lambda e: e[0])
    prev = None
    for y, x, label in ends:                            # bottom to top: push each label up if it would collide
        y_text = y if prev is None else max(y, prev + gap)
        prev = y_text
        ax.annotate(label, (x, y_text), xytext=(7, 0), textcoords="offset points", va="center", fontsize=8,
                    color="#3d3d3a", annotation_clip=False)
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)
    ax.margins(x=0.12)
    ax.legend(ncol=min(len(wide.columns), 8), loc="upper center", bbox_to_anchor=(0.5, -0.13), fontsize=8)

## 1. Annual financials
Metrics: `revenue`, `operating_income`, `net_income`, `marketing`, `sga`, `rnd` (USD, fiscal years). Not every company reports every metric. See section 5 to find others.

In [ ]:
annual = edgar.get_financials(companies.ticker.tolist(), freq="annual")
annual.to_csv(PROCESSED / "edgar_annual.csv", index=False)
print(f"{len(annual):,} rows, saved to analysis/data/processed/edgar_annual.csv")

# Wide table: one row per company-year, one column per metric, in $ billions
fin = (annual.pivot_table(index=["ticker", "fiscal_year"], columns="metric", values="value") / 1e9)
fin["operating_margin_%"] = fin["operating_income"] / fin["revenue"] * 100
fin["marketing_%_rev"] = fin["marketing"] / fin["revenue"] * 100
fin["revenue_growth_%"] = fin.groupby(level="ticker")["revenue"].pct_change() * 100
fin.loc[["BKNG", "EXPE"]].tail(12)

**Check before you cite:** non-USD filers keep their own currency (see the `unit` column in `annual`), and fiscal years differ (Intuit's ends in July). Every row keeps its `accn` (accession number) so you can trace it to the exact filing.

In [ ]:
SECTOR = "travel"          # travel | insurance | financial | subscriptions | ai_platform
FROM_YEAR = 2019

sector_t = companies.loc[companies.sector == SECTOR, "ticker"].tolist()
sub = fin.loc[fin.index.get_level_values("ticker").isin(sector_t)]
sub = sub[sub.index.get_level_values("fiscal_year") >= FROM_YEAR]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
line_chart(axes[0], sub["revenue"].unstack("ticker"), f"{SECTOR.title()}: annual revenue", "$ billions", "{:,.1f}")
line_chart(axes[1], sub["operating_margin_%"].unstack("ticker"), f"{SECTOR.title()}: operating margin", "% of revenue", "{:.0f}%")
axes[1].axhline(0, color="#3d3d3a", lw=0.8)
plt.tight_layout()
fig.savefig(FIG / f"edgar_revenue_margin_{SECTOR}.png", dpi=200, bbox_inches="tight")

## 2. Marketing intensity
Online travel agencies and comparison sites buy traffic, mostly through Google search ads. If consumers start searching and booking through AI agents, watch whether marketing as a % of revenue rises (paying more to stay visible) or the companies shift strategy.

**Caveat:** a business-to-business company like Sabre reports only a small advertising line, so it shows ~0% and isn't comparable to the consumer-facing companies.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
line_chart(ax, sub["marketing_%_rev"].unstack("ticker"), f"{SECTOR.title()}: marketing spend as % of revenue", "% of revenue", "{:.0f}%")
plt.tight_layout()
fig.savefig(FIG / f"edgar_marketing_{SECTOR}.png", dpi=200, bbox_inches="tight")

## 3. Recent quarters
Annual data lags. Quarterly data shows the latest trend. Q4 is calculated as full year minus Q1 to Q3 (marked `derived` in the `form` column). Year-over-year growth compares each quarter with the same quarter last year, which removes seasonality (travel peaks in summer). Foreign filers (e.g. Trip.com, which files a 20-F) don't file quarterly 10-Qs, so they won't appear here.

In [ ]:
quarterly = edgar.get_financials(sector_t, metrics=["revenue"], freq="quarterly")
quarterly.to_csv(PROCESSED / f"edgar_quarterly_revenue_{SECTOR}.csv", index=False)

q = quarterly.pivot_table(index="period_end", columns="ticker", values="value")
q = q[q.index >= f"{FROM_YEAR}-01-01"]
yoy = q.pct_change(4, fill_method=None) * 100

fig, ax = plt.subplots(figsize=(11, 4.5))
YOY_FROM = "2023-01-01"   # earlier quarters are distorted by the COVID collapse and rebound (+/-100%+)
line_chart(ax, yoy[yoy.index >= YOY_FROM].dropna(how="all"), f"{SECTOR.title()}: quarterly revenue growth, year over year", "% vs. same quarter last year", "{:.0f}%")
ax.axhline(0, color="#3d3d3a", lw=0.8)
plt.tight_layout()
fig.savefig(FIG / f"edgar_quarterly_yoy_{SECTOR}.png", dpi=200, bbox_inches="tight")

## 4. AI mentions in filings
Uses EDGAR full-text search to find which annual reports (10-K / 20-F) contain each phrase. This counts **filings that mention the phrase**, not how many times it appears. Click the URLs and read the context: is AI described as a *risk* (Item 1A Risk Factors) or an *opportunity*? Record quotes in `analysis/financials/fundamentals.md`.

In [ ]:
PHRASES = ["AI agents", "agentic", "generative AI", "large language model"]
MENTION_TICKERS = companies.loc[companies.role.str.contains("intermediary|challenger|adapter"), "ticker"].tolist()

mentions = edgar.search_mentions(PHRASES, MENTION_TICKERS, forms=("10-K", "20-F"), start="2019-01-01")
mentions.to_csv(PROCESSED / "edgar_ai_mentions.csv", index=False)
mentions["year"] = mentions.file_date.dt.year
print(f"{len(mentions)} filing-phrase matches")

# First year each company used each phrase in an annual report
first = mentions.groupby(["ticker", "phrase"]).year.min().unstack("phrase").reindex(columns=PHRASES)
first

In [ ]:
# Heatmap: number of the 4 phrases appearing in each company's annual report, by filing year
grid = (mentions.groupby(["ticker", "year"]).phrase.nunique().unstack("year")
        .reindex(MENTION_TICKERS).fillna(0).astype(int))

from matplotlib.colors import ListedColormap
blues = ListedColormap(["#f4f3ef", "#9ec5f4", "#5598e7", "#256abf", "#0d366b"])   # 0..4, light -> dark

fig, ax = plt.subplots(figsize=(1.0 * len(grid.columns) + 2, 0.4 * len(grid) + 1.2))
ax.imshow(grid.values, cmap=blues, vmin=0, vmax=len(PHRASES), aspect="auto")
ax.set_xticks(range(len(grid.columns)), grid.columns)
ax.set_yticks(range(len(grid)), grid.index)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
for (i, j), v in pd.DataFrame(grid.values).stack().items():
    if v:
        ax.text(j, i, v, ha="center", va="center", fontsize=8, color="white" if v >= 3 else "#1a1a19")
ax.set_title(f"How many AI phrases appear in each annual report\n({', '.join(PHRASES)})", loc="left")
plt.tight_layout()
fig.savefig(FIG / "edgar_ai_mentions_heatmap.png", dpi=200, bbox_inches="tight")

In [ ]:
# Links to read: filings that mention agent-related phrases
(mentions[mentions.phrase.isin(["AI agents", "agentic"])]
 .sort_values("file_date", ascending=False)[["ticker", "phrase", "form", "file_date", "url"]]
 .style.format({"url": lambda u: f'<a href="{u}" target="_blank">open</a>'}))

## 5. Explore & read
### Find more metrics
Companies report hundreds of XBRL tags, including some company-specific ones. Search by keyword, then add the tag to `METRICS` in `edgar.py`. Note that operating metrics like **gross bookings** and **room nights** usually aren't tagged. Get those from the filing text or the earnings press release instead.

In [ ]:
edgar.available_tags("BKNG", search="employee")

### Open the actual filings

In [ ]:
edgar.list_filings("BKNG", forms=("10-K",)).head(5)

## Takeaways
- TODO: revenue/margin story for the sector
- TODO: marketing intensity trend
- TODO: when AI agents first appeared in filings, and how they're framed (risk vs. opportunity)

Log every filing you quote in `research/sources.md`.